# **Importing Necessary Libraries**

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import NoSuchElementException, TimeoutException

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

import os
import time
import ast
import re

import numpy as np
import pandas as pd

# **Defining Utility Functions**

In [ ]:
def retrieving_html_content(driver):
  return BeautifulSoup(driver.page_source, 'lxml')

# **Setting Up Driver & Accessing Homepage**

In [ ]:
def setup_driver(driver_path: str, url: str):
  os.environ['PATH'] += driver_path

  options = webdriver.ChromeOptions()
  options.add_experimental_option("detach", True)
  options.add_experimental_option('excludeSwitches', ['enable-logging'])

  driver = webdriver.Chrome(options=options)
  driver.maximize_window()
  driver.get(url)
  driver.implicitly_wait(10)

  return driver

# **Scraping Category Links**

In [ ]:
def get_internal_links(urls, base_url):
  base_url = urlparse(base_url).scheme + "://" + urlparse(base_url).netloc
  internal_links = set()

  for href in urls:
    full_url = urljoin(base_url, href)

    if urlparse(full_url).netloc == urlparse(base_url).netloc:
      internal_links.add(full_url)

  return list(internal_links)

In [ ]:
def scrape_category_links(driver):
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

    links = []

    # Scraping category links
    menu_table = driver.find_element(By.CSS_SELECTOR, "#__layout > div > header > div.bottom-header.bg-green-gradient-2.menu-green-blue")
    menu_items = menu_table.find_elements(By.CSS_SELECTOR, ".flex-auto.menu-item.mega-menu")

    for menu_item in menu_items[1:5]:
        link_elements = menu_item.find_elements(By.CSS_SELECTOR, ".px-2.py-4 a")
        links.extend([item.get_attribute("href") for item in link_elements if item.get_attribute("href")])

    main_section = driver.find_element(By.CSS_SELECTOR, "#__layout > div > main > section > div > section:nth-child(7) > div")
    categories = main_section.find_elements(By.CSS_SELECTOR, ".bg-white.lg\\:flex.lg\\:mb-12")

    for category in categories:
        link_elements = category.find_elements(By.CSS_SELECTOR, ".filter_horizontal a")

        for link in link_elements:
            href = link.get_attribute("href")
            if href:
                links.append(href)

    category_urls = get_internal_links(set(links), 'https://rangdongstore.vn/')

    return category_urls

# **Scraping Product Links**

In [ ]:
def scrape_product_links(driver, url):
  product_links = set()

  try:
    driver.get(url)
    time.sleep(1)

    products_exist = driver.find_elements(By.CSS_SELECTOR, "div.product-content a")
    if not products_exist:
      print(f"⚠️ No products found on {url}, skipping...")
      return product_links

    WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.product-content a"))
    )

    products = driver.find_elements(By.CSS_SELECTOR, "div.product-content a")
    for product in products:
      link = product.get_attribute("href")
      product_links.add(link)

  except (NoSuchElementException, TimeoutException) as e:
    print(f"⚠️ Error scraping {url}: {e}, skipping...")

  return product_links

In [ ]:
def going_through_categories(driver, category_urls):
    grouped_links = {}

    for url in category_urls:
        try:
            driver.get(url)
            soup = retrieving_html_content(driver)

            category_name = soup.select_one("head title").text.strip() if soup.select_one("head title") else "Unknown Category"
            product_links = scrape_product_links(driver, url)  # Scrape first page
            product_links = {link for link in product_links if 'tel' not in link}

            # Extract pagination
            pagination_elements = soup.select("section.categories ul li")
            try:
                last_page = int(pagination_elements[-2].text) if len(pagination_elements) > 1 else 1
            except (IndexError, ValueError):
                last_page = 1  # Default to 1 if there's an issue parsing pagination

            print(f"🔎 Scraping category: {category_name} ({last_page} pages)")

            # Scrape remaining pages (start from page 2 to prevent duplicate scraping)
            for page in range(2, last_page + 1):
                paginated_url = f"{url}?page={page}"
                product_links.update(scrape_product_links(driver, paginated_url))  # Merge sets
                more_links = {link for link in more_links if 'tel' not in link}
            product_links.update(scrape_product_links(driver, paginated_url))  # Merge sets

            # Only add category if it contains product links
            if product_links:
                grouped_links[category_name] = list(product_links)

        except Exception as e:
            print(f"❌ Failed to scrape category {url}: {e}")

    return grouped_links

# **Scraping Product Information**

In [ ]:
def scrape_general_info(driver):
    products = {}

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, '//div[@class="aspect-w-1 aspect-h-1 w-full"]//img'))
        )

        img = driver.find_element(By.XPATH, '//div[@class="aspect-w-1 aspect-h-1 w-full"]//img')
        img_link = img.get_attribute('src')
        products['Image Link'] = img_link
    except:
        pass

    model = driver.find_elements(By.XPATH, '//*[contains(@class,"text-sm font-medium text-strong-blue-500 lg:text-base")]')
    if len(model) < 2:
        model_name = ''.join(re.findall(pattern = '[^(Model: )]', string = model[0].text))
        products['Model'] = model_name
    else:
        dong_tren = ''.join(re.findall(pattern = '[^(Model: )]', string = model[0].text))
        products['Model'] = dong_tren + ' ' + model[1].text

    name = driver.find_element(By.TAG_NAME, 'h1')
    products['Product Name'] = name.text

    try:
        try:
            price_frame = driver.find_element(By.CSS_SELECTOR, 'div[class = "flex flex-wrap items-center mb-1"]')
            prices = price_frame.find_elements(By.CSS_SELECTOR, '[class*="mr-4"]')
            if len(prices) > 1:
                products['Original Price'] = prices[1].text
                products['Sale Price'] = prices[0].text
            else:
                products['Original Price'] = prices[0].text
                products['Sale Price'] = np.nan
        except:
            original_price = driver.find_element(By.CSS_SELECTOR, 'div[class="line-through caption"]')
            sale_price = driver.find_element(By.CSS_SELECTOR, 'div[class ="font-bold h5"]')
            products['Original Price'] = original_price.text
            products['Sale Price'] = sale_price.text
    except:
        products['Original Price'] = 'Giá liên hệ'
        products['Sale Price'] = np.nan

    tskt = driver.find_element(By.ID, 'tab-info-specification')

    # Scroll into view
    driver.execute_script("arguments[0].scrollIntoView();", tskt)
    ActionChains(driver).move_to_element(tskt).click().perform()

    try:
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.TAG_NAME, 'table'))
        )

        tables = driver.find_elements(By.TAG_NAME, 'table')

        for table in tables:
            rows = table.find_elements(By.TAG_NAME, 'tr')
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, 'td')
                column_name = ' '.join(re.findall(string = cells[0].text, pattern='[\w()]+'))
                products[column_name] = cells[1].text

    except:
        pass

    return pd.DataFrame([products])

<>:62: SyntaxWarning: invalid escape sequence '\w'
<>:62: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Admin\AppData\Local\Temp\ipykernel_14376\395446927.py:62: SyntaxWarning: invalid escape sequence '\w'
  column_name = ' '.join(re.findall(string = cells[0].text, pattern='[\w()]+'))


In [ ]:
def scrape_choose_menu(driver, dataset):
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div[class="hidden lg:block"]'))
        )
        other = driver.find_element(By.CSS_SELECTOR, 'div[class="hidden lg:block"]')
        other_features = other.find_elements(By.CLASS_NAME, 'mb-3')

        len_cat = []
        all_cat = []
        caption_name = []
        for feat in other_features:
            feat_in_cat = feat.find_elements(By.CSS_SELECTOR, 'div[class*="mr-2"]')
            num_feat_cat = len(feat_in_cat)
            len_cat.append(num_feat_cat)
            all_cat.append(feat_in_cat)
            caption = feat.find_element(By.CSS_SELECTOR, 'div[class = "mb-2 caption"]')
            caption_name.append(' '.join(re.findall(string = caption.text, pattern='([\w\s]+):')))

        len_cat_for_prod = [np.arange(i) for i in len_cat]
        cat_index = list(product(*len_cat_for_prod))

        for combin in cat_index:
            combin = list(combin)
            for i in range(len(combin)):
                feat_chosen = all_cat[i][combin[i]]
                ActionChains(driver).move_to_element(feat_chosen).click().perform()
                chosen_cat_info = feat_chosen.find_element(By.CSS_SELECTOR, 'div[class *= "relative"]').text
                dataset[f"{caption_name[i]}"] = chosen_cat_info
            new_product = scrape_general_info(driver)
            dataset = pd.concat([dataset, new_product], join = 'outer', ignore_index = True)
    except:
        new_product = scrape_general_info(driver)
        dataset = pd.concat([dataset, new_product], join = 'outer', ignore_index = True)

    return dataset

<>:18: SyntaxWarning: invalid escape sequence '\w'
<>:18: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Admin\AppData\Local\Temp\ipykernel_14376\2270553284.py:18: SyntaxWarning: invalid escape sequence '\w'
  caption_name.append(' '.join(re.findall(string = caption.text, pattern='([\w\s]+):')))


In [ ]:
def scrape_products(driver, product_links):
    category = pd.DataFrame()

    for prod_link in product_links:
        driver.get(prod_link)
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[class="mb-4"]'))
            )

            cong_suat = driver.find_element(By.CSS_SELECTOR, 'div[class="mb-4"]')
            cong_suat_element = cong_suat.find_elements(By.TAG_NAME, 'a')
            cong_suat_link = [i.get_attribute('href') for i in cong_suat_element if 'chinh-sach' not in i.get_attribute('href')]

            if len(cong_suat_link) == 0:
                raise ValueError

            for link in cong_suat_link:
                driver.get(link)
                try:
                    category = scrape_choose_menu(driver, category)
                except:
                    print(f"⚠️ Error scraping {link}, skipping...")
                    pass
        except:
            try:
                category = scrape_choose_menu(driver, category)
            except:
                print(f'Error scraping ({prod_link}), skipping...')
                pass

    time.sleep(5)

    category.drop_duplicates(inplace = True)
    return category

# **Export**

In [ ]:
def scrape_and_export(driver, data_dict):
    for category_name, links in data_dict.items():
        print(f"Scraping category: {category_name}")
        category_data = scrape_products(driver, links)

        filename = f"{category_name.replace(' ', '_').replace('/', '_')}.csv"
        category_data.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"Saved: {filename}")

        time.sleep(45)

# **Taking Product Links from CSV**

In [ ]:
def taking_prod_links(df):
    product_dict = {}

    #Chuyển toàn bộ
    for i in range(int(df.shape[1])):
        lst = df.iloc[:, i].apply(ast.literal_eval)
        product_dict[df.columns[i]] = lst[0]

    return product_dict


# **Main Scraping Logic**

In [ ]:
# Setup WebDriver
driver = setup_driver(r'D:/DAAI Lab/Web Scraping Learning/chromedriver.exe', 'https://rangdongstore.vn/')

# Handling website redirection and popup ad
try:
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//div[@class="aside__content md:text-center max-md:px-2"]//button')))
    dan_dung = driver.find_element(By.XPATH, '//div[@class="aside__content md:text-center max-md:px-2"]//button')
    dan_dung.click()
except TimeoutException:
    print("No popup button found.")

try:
    WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, 'close-button-1454703513200')))
    popup_button = driver.find_element(By.ID, 'close-button-1454703513200')
    popup_button.click()
except TimeoutException:
    print("No popup found.")

# category_urls = scrape_category_links(driver)
# grouped_links = going_through_categories(driver, category_urls)

# In case you already had the scraped product links, please run the below code
df = pd.read_csv("product_links.csv")
prods = df.iloc[:, 45:57]
grouped_links = taking_prod_links(prods)

scrape_and_export(driver, grouped_links)
driver.quit()

Scraping category: Đèn Led Ray
                                           Image Link  \
0   https://static.rangdongstore.vn/240130017071/2...   
1   https://static.rangdongstore.vn/240130017076/2...   
2   https://static.rangdongstore.vn/231227016266/2...   
3   https://static.rangdongstore.vn/240130017072/2...   
4   https://static.rangdongstore.vn/231227016258/2...   
5   https://static.rangdongstore.vn/240130017070/2...   
6   https://static.rangdongstore.vn/240123016756/2...   
7   https://static.rangdongstore.vn/240130017075/2...   
8   https://static.rangdongstore.vn/240130017073/2...   
9   https://static.rangdongstore.vn/240123016754/2...   
10  https://static.rangdongstore.vn/240123016747/2...   
11  https://static.rangdongstore.vn/240130017074/2...   

                           Model  \
0       RAYLED.48/NG LED00066793   
1   DR-RAYLED.48200W LED00066798   
2    RLT02670/20W48V LED00063606   
3      RAYLED.48/NXG LED00066794   
4    RLT02370/10W48V LED00063605   
5       RAY